In [ ]:
!pip install bertopic sentence-transformers hdbscan umap-learn gspread

import pandas as pd
import gspread
import re
import random
from google.colab import auth, drive
from google.auth import default
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
SPREADSHEET_ID = "1JFpAxDlJhM6UhL3u5ZuFOoIGvlf5AuG7qNhrRygIL3k"
WORKSHEET_NAME = "Community_Survey"

TARGET_COLS = [
    "Q2A - If you are dissatisfied with any of the items listed in Quality of Life, why?",
    "Q4A - If you are dissatisfied with any of the items listed in Economic Opportunity and Affordability, why?",
    "Q6A - If you are dissatisfied with any of the items listed in Health and Environment, why?",
    "Q8A - If you are dissatisfied with any of the items listed in Safety, why?",
    "Q10A - If you are dissatisfied with any of the items listed in Mobility, why?",
    "Q12A - If you are dissatisfied with any of the items listed in Culture and Lifelong Learning, why?",
    "Q14A - If you are dissatisfied with any of the items listed in Government that Works for All, why?",
    "Q25 - If there was one thing you could share with the Mayor regarding the City of Austin (any comment, suggestion, etc.), what would it be?",
]

RANDOM_SEED = 42

def _is_nonempty(x) -> bool:
    if x is None:
        return False
    s = str(x).strip()
    return s != "" and s.lower() != "nan"

def load_sheet_to_long_df(spreadsheet_id, worksheet_name, target_cols):
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(spreadsheet_id)
    ws = sh.worksheet(worksheet_name)
    records = ws.get_all_records()
    df = pd.DataFrame(records)
    missing = [c for c in target_cols if c not in df.columns]
    if missing:
        raise KeyError("Missing columns:\n- " + "\n- ".join(missing))
    df = df.copy()
    df["source_row"] = df.index + 2
    long_parts = []
    for col in target_cols:
        part = df[["source_row", col]].copy()
        part = part.rename(columns={col: "comment_text"})
        part["question_col"] = col
        part["comment_text"] = part["comment_text"].astype(str).map(lambda s: s.strip())
        part = part[part["comment_text"].map(_is_nonempty)]
        long_parts.append(part)
    long_df = pd.concat(long_parts, ignore_index=True)
    long_df = long_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    return long_df

long_df = load_sheet_to_long_df(SPREADSHEET_ID, WORKSHEET_NAME, TARGET_COLS)
print("Rows:", len(long_df))

Rows: 13542


In [ ]:
def clean_comment(text):
    text = text.strip()
    if text.isupper():
        text = text.capitalize()
    text = re.sub(r"\s+", " ", text)
    return text

docs = long_df["comment_text"].astype(str).map(clean_comment).tolist()

drive.mount('/content/drive')
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic.load("/content/drive/MyDrive/bertopic_austin_survey", embedding_model=embedding_model)
print(f"Model loaded — {len(topic_model.get_topic_info()) - 1} topics")

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded — 49 topics


In [ ]:
import random
import re

random.seed(42)

current_topics = topic_model.topics_
comments = docs  # already cleaned

# --- Build index with metadata ---
records = []
for i, (doc, topic) in enumerate(zip(comments, current_topics)):
    word_count = len(doc.split())
    records.append({"idx": i, "text": doc, "topic": topic, "word_count": word_count})

# --- Strategy 1: Random sample (should yield mostly class 1 and 2) ---
random_pool = [r for r in records]
random.shuffle(random_pool)
random_sample = random_pool[:200]

# --- Strategy 2: Keyword-enriched sample (likely class 3) ---
action_patterns = re.compile(
    r"\b(should|need to|fix|add|build|remove|install|replace|retime|expand|widen|"
    r"extend|enforce|improve|between|intersection|near|block|street|lane|road|"
    r"boulevard|blvd|ave|drive|creek|park)\b",
    re.IGNORECASE
)
number_pattern = re.compile(r"\b\d{3,5}\b")  # addresses, dollar amounts

keyword_pool = []
for r in records:
    matches = len(action_patterns.findall(r["text"]))
    has_number = bool(number_pattern.search(r["text"]))
    if matches >= 2 or (matches >= 1 and has_number):
        keyword_pool.append(r)

# Remove any already in random sample
random_idxs = {r["idx"] for r in random_sample}
keyword_pool = [r for r in keyword_pool if r["idx"] not in random_idxs]
random.shuffle(keyword_pool)
keyword_sample = keyword_pool[:200]

# --- Strategy 3: Topic-enriched from operationally specific topics ---
specific_topics = {21, 24, 25, 32, 33, 34, 36, 37, 42, 45, 47}  # permitting, lighting, potholes, etc.
used_idxs = random_idxs | {r["idx"] for r in keyword_sample}

topic_pool = [r for r in records if r["topic"] in specific_topics and r["idx"] not in used_idxs]
# Prefer longer comments within these topics
topic_pool.sort(key=lambda r: r["word_count"], reverse=True)
topic_sample = topic_pool[:200]

# --- Combine and summarize ---
all_samples = random_sample + keyword_sample + topic_sample

# Check for any remaining duplicates
all_idxs = [r["idx"] for r in all_samples]
assert len(all_idxs) == len(set(all_idxs)), "Duplicates found!"

print(f"Random sample:   {len(random_sample)}")
print(f"Keyword sample:  {len(keyword_sample)}")
print(f"Topic sample:    {len(topic_sample)}")
print(f"Total to label:  {len(all_samples)}")
print(f"Unique comments: {len(set(all_idxs))}")


# --- Preview each pool ---
for name, sample in [("RANDOM", random_sample), ("KEYWORD", keyword_sample), ("TOPIC", topic_sample)]:
    print(f"\n{'='*60}")
    print(f"  {name} SAMPLE PREVIEW")
    print(f"{'='*60}")
    preview = random.sample(sample, min(8, len(sample)))
    for r in preview:
        print(f"  T{r['topic']:>2} ({r['word_count']:>3}w) {r['text'][:100]}")

Random sample:   200
Keyword sample:  200
Topic sample:    200
Total to label:  600
Unique comments: 600

  RANDOM SAMPLE PREVIEW
  T 0 ( 74w) Austin is not even close to a cultural mecca. It's all music and nightlife, but we don't have a venu
  T 9 (  2w) Transportation nightmare
  T27 (  3w) Constant traffic slowdowns.
  T24 ( 13w) We were promised more lights in our neighborhood and they were never provided.
  T 3 (  4w) Please make traffic better
  T 0 ( 24w) I wish there was a solution to the homeless camping under bridges, it seems to be out of control and
  T 3 (  6w) Traffic is terrible and getting worse.
  T41 (127w) Transport all these bums to a different city and get our transportation highway issue fixed. One lan

  KEYWORD SAMPLE PREVIEW
  T16 ( 35w) There no viable public transportation options from FM 1626 or Onion Creek. I could use the bus at So
  T41 ( 97w) Hand the roads over to private companies....please. Look.. We're paying about 50 cents per gallon to
  T 6 (233w

In [ ]:
# --- Labeling interface (blinded & randomized) ---
import json
import os
import random
import textwrap

LABEL_PATH = "/content/drive/MyDrive/labeling_progress.json"

# Prepare labeling queue
label_queue = []
for source, sample in [("random", random_sample), ("keyword", keyword_sample), ("topic", topic_sample)]:
    for r in sample:
        label_queue.append({"idx": r["idx"], "text": r["text"], "topic": r["topic"], "source": source})

# Shuffle so grader can't infer source from ordering
random.seed(42)  # fixed seed for reproducibility across sessions
random.shuffle(label_queue)

# Load any existing progress
if os.path.exists(LABEL_PATH):
    with open(LABEL_PATH, "r") as f:
        labels = json.load(f)
    print(f"Resuming — {len(labels)} already labeled, {len(label_queue) - len(labels)} remaining")
else:
    labels = {}
    print(f"Starting fresh — {len(label_queue)} comments to label")

print("\nLabeling guide:")
print("  1 = Noise (vague, single-word, no specifics)")
print("  2 = Directional (topic + stance, but not operational)")
print("  3 = Actionable (specific problem, location, or suggestion)")
print("  s = Skip    q = Save & quit\n")

for item in label_queue:
    key = str(item["idx"])
    if key in labels:
        continue

    # Blinded display: no source or topic shown
    print(f"[{len(labels)+1}/{len(label_queue)}]")
    wrapped = textwrap.fill(item["text"], width=90, initial_indent="  ", subsequent_indent="  ")
    print(wrapped)
    print()

    while True:
        choice = input("Label (1/2/3/s/q): ").strip().lower()
        if choice in ("1", "2", "3"):
            # Source & topic still saved for later analysis, just hidden from grader
            labels[key] = {"label": int(choice), "source": item["source"], "topic": item["topic"]}
            break
        elif choice == "s":
            break
        elif choice == "q":
            with open(LABEL_PATH, "w") as f:
                json.dump(labels, f)
            print(f"Saved {len(labels)} labels to Drive")
            break
    if choice == "q":
        break

# Auto-save at end
with open(LABEL_PATH, "w") as f:
    json.dump(labels, f)
print(f"\nTotal labeled: {len(labels)}")

Resuming — 600 already labeled, 0 remaining

Labeling guide:
  1 = Noise (vague, single-word, no specifics)
  2 = Directional (topic + stance, but not operational)
  3 = Actionable (specific problem, location, or suggestion)
  s = Skip    q = Save & quit


Total labeled: 600


# Sonnet Labeling (Silver Set)
## Goal A: Validate Sonnet consistency — compare Sonnet silver-run labels to Sonnet gold-run labels on the 600 gold-set comments
## Goal B: Label an additional 1800 comments (silver set) with Sonnet


In [ ]:
## ============================================================
## CELL 7: Build Gold Set (600) and Silver Set (1800)
## ============================================================
## Gold set = the same 600 comments human-labeled and Sonnet-labeled
## Silver set = 1800 randomly sampled from the remaining comments

import random

# Gold set: same 600 from the three sampling strategies
gold_set = []
for source, sample in [("random", random_sample), ("keyword", keyword_sample), ("topic", topic_sample)]:
    for r in sample:
        gold_set.append({
            "idx": r["idx"],
            "text": r["text"],
            "topic": r["topic"],
            "source": source,
            "set": "gold",
        })

gold_idxs = {r["idx"] for r in gold_set}

# Silver set: random 1800 from the rest of the corpus
remaining_pool = [r for r in records if r["idx"] not in gold_idxs]
random.seed(42)
random.shuffle(remaining_pool)
silver_set = []
for r in remaining_pool[:1800]:
    silver_set.append({
        "idx": r["idx"],
        "text": r["text"],
        "topic": r["topic"],
        "source": "silver_random",
        "set": "silver",
    })

print(f"Gold set:   {len(gold_set)} comments (same 600 as human + Sonnet)")
print(f"Silver set: {len(silver_set)} comments (new random sample)")
print(f"Total for Sonnet labeling: {len(gold_set) + len(silver_set)}")
print(f"Remaining unlabeled: {len(remaining_pool) - len(silver_set)}")


Gold set:   600 comments (same 600 as human + Sonnet)
Silver set: 1800 comments (new random sample)
Total for Sonnet labeling: 2400
Remaining unlabeled: 11142


In [ ]:
## ============================================================
## CELL 8: Sonnet API Labeling (Gold + Silver)
## ============================================================
## Uses claude-sonnet-4-20250514 to label all 2400 comments.
## Saves to a separate file so original gold-set Sonnet labels
## are never overwritten.

!pip install anthropic

import json
import os
import re
import time
import random
import anthropic

# --- API key ---
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()

# --- Paths ---
SONNET_SILVER_LABEL_PATH = "/content/drive/MyDrive/claude_labels_sonnet_silver.json"

# --- Build full labeling queue: gold first, then silver ---
sonnet_silver_queue = gold_set + silver_set

# Shuffle for consistent ordering
random.seed(42)
random.shuffle(sonnet_silver_queue)

# --- Resume from checkpoint if exists ---
if os.path.exists(SONNET_SILVER_LABEL_PATH):
    with open(SONNET_SILVER_LABEL_PATH, "r") as f:
        sonnet_silver_labels = json.load(f)
    already_labeled = {k for k, v in sonnet_silver_labels.items() if v.get("label") is not None}
    remaining = [item for item in sonnet_silver_queue if str(item["idx"]) not in already_labeled]
    print(f"Sonnet silver labels file found — {len(already_labeled)} already labeled, {len(remaining)} remaining")
else:
    sonnet_silver_labels = {}
    remaining = sonnet_silver_queue
    print(f"Starting fresh Sonnet labeling — {len(remaining)} comments")

# --- Same classification prompt as original Sonnet (v3) ---
SYSTEM_PROMPT_V3 = """You are classifying public civic survey comments into one of three categories based on their actionability. Read each comment carefully and assign exactly one label.

CATEGORIES:

1 — Noise: The comment is vague, single-word, pure venting, or a meta-comment about the survey itself with no substantive content. There is no identifiable topic stance or useful information.

Examples of 1:
- "Traffic"
- "Expensive"
- "No answer at this time"
- "I hope you are voted out"

2 — Directional: The comment identifies a topic and takes a stance or expresses a clear opinion, but lacks the combination of location and problem needed to act. A city department reading this comment would understand the concern but could not dispatch someone or open an investigation without asking follow-up questions.

Examples of 2:
- "Property taxes are too high"
- "We need more public transit"
- "The permitting process is broken and took forever for my renovation"
- "Traffic on IH-35 is getting worse every year" (no actionable problem — traffic congestion on a major highway is a known general condition)
- "We were required to install a new water meter at a cost of over $30k in our older home in central Austin" (personal cost grievance with no specific location or infrastructure issue to investigate)
- "COA should stop toll road proliferation. Capital Metro should offer a fixed rate monthly ride share. Consider bicycle licensing." (policy proposals requiring legislative action, not departmental work)
- "I have seen corroded street light poles fall into the street" (real problem, but no location given — cannot be investigated)

3 — Actionable: The comment describes a problem or implies a needed action AND provides enough location detail that a city employee could find the place on a map without asking "where?" The action does not need to be explicitly stated — if someone describes trash on a named street, the implied action (clean it up) is clear.

Examples of 3:
- "Retime the traffic lights at the intersection of Lamar and 5th"
- "Givens Pool was closed for repairs and is still leaking"
- "The rail crossing arms at the MLK station stay down for 5+ minutes with no train"
- "Grass in the medians on Spicewood Springs Road is so high we've nearly had collisions due to visibility"
- "Boulder Street in 78726 is not compliant with the city's lighting regulations"
- "Bull Creek Park is overrun with trash after holiday weekends — put extra park workers on cleanup"

THE ACTIONABILITY TEST — both parts must be present:

Part 1 — Can you find it on a map?
The comment must name a place specific enough that a city employee could locate it: a street name, intersection, park, facility, address, zip code paired with a street, or similar. The location does not need to be a precise address — a named street, a named park, or a recognizable area smaller than a neighborhood is sufficient.

What fails this test: broad regions ("central Austin," "East Austin," "downtown"), unnamed personal locations ("my house," "my neighborhood," "near my office"), and situations where a real problem is described but no location is given at all.

Note: Major highways and well-known roads (IH-35, MoPac, 183, Lamar, Congress) ARE valid locations when paired with a problem that can be physically investigated at that location (potholes, lighting, debris, unsafe infrastructure). They fail this test only when the complaint is about an inherent systemic condition of the road (general traffic congestion, commute times, toll policy) that cannot be addressed by sending someone to a location.

Part 2 — Is there something to do?
The comment must describe a problem, deficiency, or need that a city department could investigate or act on within its existing operational authority. The action can be explicit ("retime the lights") or implied ("trash everywhere in Gillis Park" implies cleanup). Policy proposals that require council votes, new legislation, or strategic planning do not count — the action must be something a department could scope and begin.

What fails this test: pure opinions with no underlying operational problem, policy proposals ("offer municipal ride-sharing," "license bicycles," "lower property taxes"), and personal experiences or costs that reflect policy disagreements rather than fixable conditions.

DECISION RULES:
- When in doubt between 1 and 2: does the comment identify a real topic and take a position? If yes, it is a 2.
- When in doubt between 2 and 3: apply the two-part test strictly. Are BOTH a findable location AND an actionable problem present? If either is missing, it is a 2.
- Read the ENTIRE comment before deciding. If any part of a multi-topic comment passes both parts of the test, the whole comment is a 3.
- A comment that mentions multiple issues — some actionable, some not — is a 3 based on its most actionable element.
- Length, detail, and thoughtfulness do not determine the label. A short comment with a street name and a problem can be a 3. A long comment full of policy ideas with no locatable problem is a 2.

Respond with ONLY a JSON object: {"label": <1|2|3>, "justification": "<one sentence>"}
Do not include any other text."""

# --- Run API calls ---
BATCH_SAVE_EVERY = 50
errors = []

if len(remaining) > 0:
    for i, item in enumerate(remaining):
        key = str(item["idx"])
        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=150,
                system=SYSTEM_PROMPT_V3,
                messages=[
                    {"role": "user", "content": f'Comment: "{item["text"]}"'}
                ],
            )
            raw = response.content[0].text.strip()

            # Parse JSON response
            parsed = json.loads(raw)
            label = int(parsed["label"])
            justification = parsed.get("justification", "")

            sonnet_silver_labels[key] = {
                "label": label,
                "justification": justification,
                "source": item["source"],
                "topic": item["topic"],
                "set": item["set"],
            }

            if (i + 1) % 100 == 0:
                print(f"  [{i+1}/{len(remaining)}] labeled")

        except json.JSONDecodeError:
            # Strip markdown code fences and retry
            cleaned = re.sub(r"```json\s*", "", raw)
            cleaned = re.sub(r"```\s*", "", cleaned).strip()
            try:
                parsed = json.loads(cleaned)
                sonnet_silver_labels[key] = {
                    "label": int(parsed["label"]),
                    "justification": parsed.get("justification", ""),
                    "source": item["source"],
                    "topic": item["topic"],
                    "set": item["set"],
                }
            except (json.JSONDecodeError, KeyError, ValueError):
                match = re.match(r"(\d)", raw)
                if match:
                    sonnet_silver_labels[key] = {
                        "label": int(match.group(1)),
                        "justification": raw,
                        "source": item["source"],
                        "topic": item["topic"],
                        "set": item["set"],
                    }
                else:
                    errors.append({"idx": item["idx"], "raw": raw})
                    print(f"  ⚠ Parse error idx {item['idx']}: {raw[:80]}")

        except Exception as e:
            errors.append({"idx": item["idx"], "error": str(e)})
            print(f"  ⚠ API error idx {item['idx']}: {e}")

        # Save checkpoint
        if (i + 1) % BATCH_SAVE_EVERY == 0:
            with open(SONNET_SILVER_LABEL_PATH, "w") as f:
                json.dump(sonnet_silver_labels, f)

        time.sleep(0.5)  # Rate-limit buffer for Sonnet

    # Final save
    with open(SONNET_SILVER_LABEL_PATH, "w") as f:
        json.dump(sonnet_silver_labels, f)

    print(f"\nDone — {len(sonnet_silver_labels)} Sonnet labels saved to Drive")
    if errors:
        print(f"  {len(errors)} errors encountered")
else:
    print("All comments already labeled by Sonnet. Skipping API calls.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 1.7 MB/s eta 0:00:00
Starting fresh Sonnet labeling — 2400 comments
  [100/2400] labeled
  [200/2400] labeled
  [300/2400] labeled
  [400/2400] labeled
  [500/2400] labeled
  [600/2400] labeled
  [700/2400] labeled
  [800/2400] labeled
  [900/2400] labeled
  [1000/2400] labeled
  [1100/2400] labeled
  [1200/2400] labeled
  [1300/2400] labeled
  [1400/2400] labeled
  [1500/2400] labeled
  [1600/2400] labeled
  [1700/2400] labeled
  [1800/2400] labeled
  [1900/2400] labeled
  [2000/2400] labeled
  [2100/2400] labeled
  [2200/2400] labeled
  [2300/2400] labeled
  [2400/2400] labeled

Done — 2400 Sonnet labels saved to Drive


In [ ]:
## ============================================================
## CELL 9: Sonnet Self-Consistency Check (Gold Set, n=600)
## ============================================================
## Compares the NEW Sonnet silver-run labels to the ORIGINAL Sonnet
## gold-run labels on the same 600 comments. This measures Sonnet's
## test-retest reliability (same model, same prompt, two runs).

import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

CLAUDE_LABEL_PATH        = "/content/drive/MyDrive/claude_labels_v3.json"           # Sonnet gold run
SONNET_SILVER_LABEL_PATH = "/content/drive/MyDrive/claude_labels_sonnet_silver.json" # Sonnet silver run

with open(CLAUDE_LABEL_PATH, "r") as f:
    sonnet_gold_labels = json.load(f)

with open(SONNET_SILVER_LABEL_PATH, "r") as f:
    sonnet_silver_labels = json.load(f)

# Gold set keys only (the 600 that both runs cover)
gold_keys = sorted(set(sonnet_gold_labels.keys()) & set(sonnet_silver_labels.keys()))
print(f"Gold set overlap (Sonnet gold-run ∩ Sonnet silver-run): {len(gold_keys)}")

gold_vals  = [sonnet_gold_labels[k]["label"]   for k in gold_keys]
silver_vals = [sonnet_silver_labels[k]["label"] for k in gold_keys]

# ==============================================================
# BINARY: Not-Actionable (1+2) vs Actionable (3)
# ==============================================================
print(f"\n{'='*60}")
print("  BINARY: Sonnet Gold-Run vs Sonnet Silver-Run")
print(f"{'='*60}\n")

gold_binary   = [1 if v == 3 else 0 for v in gold_vals]
silver_binary = [1 if v == 3 else 0 for v in silver_vals]

binary_kappa = cohen_kappa_score(gold_binary, silver_binary)
binary_agree = np.mean(np.array(gold_binary) == np.array(silver_binary))

print(f"Cohen's Kappa:    {binary_kappa:.3f}")
print(f"Raw agreement:    {binary_agree:.1%}")

if binary_kappa >= 0.8:
    interp = "Almost perfect"
elif binary_kappa >= 0.6:
    interp = "Substantial"
elif binary_kappa >= 0.4:
    interp = "Moderate"
elif binary_kappa >= 0.2:
    interp = "Fair"
else:
    interp = "Slight"
print(f"Interpretation:   {interp} agreement\n")

binary_cm = confusion_matrix(gold_binary, silver_binary, labels=[0, 1])
binary_cm_df = pd.DataFrame(
    binary_cm,
    index=["GoldRun=Not-Actionable", "GoldRun=Actionable"],
    columns=["SilverRun=Not-Actionable", "SilverRun=Actionable"],
)
print(f"Confusion Matrix:\n{binary_cm_df}\n")

print("Classification Report (Gold-run as reference):")
print(classification_report(
    gold_binary, silver_binary,
    labels=[0, 1],
    target_names=["Not-Actionable (1+2)", "Actionable (3)"],
))

# ==============================================================
# FULL 3-CLASS
# ==============================================================
print(f"{'='*60}")
print("  FULL 3-CLASS: Sonnet Gold-Run vs Sonnet Silver-Run")
print(f"{'='*60}\n")

kappa = cohen_kappa_score(gold_vals, silver_vals)
weighted_kappa = cohen_kappa_score(gold_vals, silver_vals, weights="quadratic")
raw_agree = np.mean(np.array(gold_vals) == np.array(silver_vals))

print(f"Cohen's Kappa (unweighted):        {kappa:.3f}")
print(f"Cohen's Kappa (quadratic-weighted): {weighted_kappa:.3f}")
print(f"Raw agreement:                      {raw_agree:.1%}\n")

labels_list = [1, 2, 3]
cm = confusion_matrix(gold_vals, silver_vals, labels=labels_list)
cm_df = pd.DataFrame(
    cm,
    index=[f"GoldRun={l}" for l in labels_list],
    columns=[f"SilverRun={l}" for l in labels_list],
)
print(f"Confusion Matrix:\n{cm_df}\n")

print("Classification Report (Gold-run as reference):")
print(classification_report(
    gold_vals, silver_vals,
    labels=labels_list,
    target_names=["1-Noise", "2-Directional", "3-Actionable"],
))


Gold set overlap (Sonnet gold-run ∩ Sonnet silver-run): 598

  BINARY: Sonnet Gold-Run vs Sonnet Silver-Run

Cohen's Kappa:    0.970
Raw agreement:    99.3%
Interpretation:   Almost perfect agreement

Confusion Matrix:
                        SilverRun=Not-Actionable  SilverRun=Actionable
GoldRun=Not-Actionable                       520                     2
GoldRun=Actionable                             2                    74

Classification Report (Gold-run as reference):
                      precision    recall  f1-score   support

Not-Actionable (1+2)       1.00      1.00      1.00       522
      Actionable (3)       0.97      0.97      0.97        76

            accuracy                           0.99       598
           macro avg       0.98      0.98      0.98       598
        weighted avg       0.99      0.99      0.99       598

  FULL 3-CLASS: Sonnet Gold-Run vs Sonnet Silver-Run

Cohen's Kappa (unweighted):        0.948
Cohen's Kappa (quadratic-weighted): 0.957
Raw agre

In [ ]:
## ============================================================
## CELL 10: Sonnet (Silver-Run) vs Corrected Human Labels (Gold Set)
## ============================================================
## Uses the EXISTING human labels + audit corrections (unchanged).
## Compares the new Sonnet silver-run to corrected human labels.
## (This parallels the original Sonnet gold-run vs Human comparison.)

import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

LABEL_PATH               = "/content/drive/MyDrive/labeling_progress.json"
AUDIT_PATH               = "/content/drive/MyDrive/audit_labels_v3.json"
SONNET_SILVER_LABEL_PATH = "/content/drive/MyDrive/claude_labels_sonnet_silver.json"

with open(LABEL_PATH, "r") as f:
    human_labels = json.load(f)

with open(AUDIT_PATH, "r") as f:
    audit_labels = json.load(f)

with open(SONNET_SILVER_LABEL_PATH, "r") as f:
    sonnet_silver_labels = json.load(f)

# Build corrected human labels (same logic as original notebook)
common_keys = sorted(set(human_labels.keys()) & set(sonnet_silver_labels.keys()))
print(f"Overlap (Corrected Human ∩ Sonnet silver-run): {len(common_keys)}")

corrected_vals = []
sonnet_s_vals = []

for k in common_keys:
    if k in audit_labels:
        corrected_vals.append(audit_labels[k]["audit_label"])
    else:
        corrected_vals.append(human_labels[k]["label"])
    sonnet_s_vals.append(sonnet_silver_labels[k]["label"])

# --- Binary ---
print(f"\n{'='*60}")
print("  BINARY: Sonnet (Silver-Run) vs Corrected Human")
print(f"{'='*60}\n")

corrected_binary = [1 if v == 3 else 0 for v in corrected_vals]
sonnet_s_binary  = [1 if v == 3 else 0 for v in sonnet_s_vals]

binary_kappa = cohen_kappa_score(corrected_binary, sonnet_s_binary)
binary_agree = np.mean(np.array(corrected_binary) == np.array(sonnet_s_binary))

print(f"Cohen's Kappa:    {binary_kappa:.3f}")
print(f"Raw agreement:    {binary_agree:.1%}")

if binary_kappa >= 0.8:
    interp = "Almost perfect"
elif binary_kappa >= 0.6:
    interp = "Substantial"
elif binary_kappa >= 0.4:
    interp = "Moderate"
elif binary_kappa >= 0.2:
    interp = "Fair"
else:
    interp = "Slight"
print(f"Interpretation:   {interp} agreement\n")

binary_cm = confusion_matrix(corrected_binary, sonnet_s_binary, labels=[0, 1])
binary_cm_df = pd.DataFrame(
    binary_cm,
    index=["Human=Not-Actionable", "Human=Actionable"],
    columns=["Sonnet=Not-Actionable", "Sonnet=Actionable"],
)
print(f"Confusion Matrix:\n{binary_cm_df}\n")

# --- 3-Class ---
print(f"{'='*60}")
print("  FULL 3-CLASS: Sonnet (Silver-Run) vs Corrected Human")
print(f"{'='*60}\n")

kappa_3 = cohen_kappa_score(corrected_vals, sonnet_s_vals)
weighted_kappa_3 = cohen_kappa_score(corrected_vals, sonnet_s_vals, weights="quadratic")
raw_agree_3 = np.mean(np.array(corrected_vals) == np.array(sonnet_s_vals))

print(f"Cohen's Kappa (unweighted):        {kappa_3:.3f}")
print(f"Cohen's Kappa (quadratic-weighted): {weighted_kappa_3:.3f}")
print(f"Raw agreement:                      {raw_agree_3:.1%}\n")

labels_list = [1, 2, 3]
cm = confusion_matrix(corrected_vals, sonnet_s_vals, labels=labels_list)
cm_df = pd.DataFrame(
    cm,
    index=[f"Human={l}" for l in labels_list],
    columns=[f"Sonnet={l}" for l in labels_list],
)
print(f"Confusion Matrix:\n{cm_df}\n")

print("Classification Report (Human as reference):")
print(classification_report(
    corrected_vals, sonnet_s_vals,
    labels=labels_list,
    target_names=["1-Noise", "2-Directional", "3-Actionable"],
))


Overlap (Corrected Human ∩ Sonnet silver-run): 600

  BINARY: Sonnet (Silver-Run) vs Corrected Human

Cohen's Kappa:    0.862
Raw agreement:    96.8%
Interpretation:   Almost perfect agreement

Confusion Matrix:
                      Sonnet=Not-Actionable  Sonnet=Actionable
Human=Not-Actionable                    511                  6
Human=Actionable                         13                 70

  FULL 3-CLASS: Sonnet (Silver-Run) vs Corrected Human

Cohen's Kappa (unweighted):        0.456
Cohen's Kappa (quadratic-weighted): 0.590
Raw agreement:                      68.7%

Confusion Matrix:
         Sonnet=1  Sonnet=2  Sonnet=3
Human=1        62       169         0
Human=2         0       280         6
Human=3         0        13        70

Classification Report (Human as reference):
               precision    recall  f1-score   support

      1-Noise       1.00      0.27      0.42       231
2-Directional       0.61      0.98      0.75       286
 3-Actionable       0.92      0.84 

In [ ]:
## ============================================================
## CELL 11: Comparison Summary — Human vs Sonnet (Both Runs)
## ============================================================
## Shows pairwise kappas: Human vs Sonnet gold-run, Human vs Sonnet
## silver-run, and Sonnet gold-run vs silver-run (self-consistency).

import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

LABEL_PATH               = "/content/drive/MyDrive/labeling_progress.json"
AUDIT_PATH               = "/content/drive/MyDrive/audit_labels_v3.json"
CLAUDE_LABEL_PATH        = "/content/drive/MyDrive/claude_labels_v3.json"
SONNET_SILVER_LABEL_PATH = "/content/drive/MyDrive/claude_labels_sonnet_silver.json"

with open(LABEL_PATH, "r") as f:
    human_labels = json.load(f)
with open(AUDIT_PATH, "r") as f:
    audit_labels = json.load(f)
with open(CLAUDE_LABEL_PATH, "r") as f:
    sonnet_gold_labels = json.load(f)
with open(SONNET_SILVER_LABEL_PATH, "r") as f:
    sonnet_silver_labels = json.load(f)

# Common keys across all three
common_keys = sorted(
    set(human_labels.keys()) & set(sonnet_gold_labels.keys()) & set(sonnet_silver_labels.keys())
)
print(f"Three-way overlap: {len(common_keys)} comments\n")

# Build arrays
corrected = []
sonnet_g = []
sonnet_s = []
for k in common_keys:
    corrected.append(audit_labels[k]["audit_label"] if k in audit_labels else human_labels[k]["label"])
    sonnet_g.append(sonnet_gold_labels[k]["label"])
    sonnet_s.append(sonnet_silver_labels[k]["label"])

corrected = np.array(corrected)
sonnet_g = np.array(sonnet_g)
sonnet_s = np.array(sonnet_s)

# Binary versions
corrected_b = (corrected == 3).astype(int)
sonnet_g_b = (sonnet_g == 3).astype(int)
sonnet_s_b = (sonnet_s == 3).astype(int)

# Pairwise kappas
pairs = [
    ("Human vs Sonnet (gold-run)",   corrected, sonnet_g,  corrected_b, sonnet_g_b),
    ("Human vs Sonnet (silver-run)", corrected, sonnet_s,  corrected_b, sonnet_s_b),
    ("Sonnet gold vs silver (self)", sonnet_g,  sonnet_s,  sonnet_g_b,  sonnet_s_b),
]

rows = []
for name, a3, b3, ab, bb in pairs:
    rows.append({
        "Pair": name,
        "3-Class Kappa": f"{cohen_kappa_score(a3, b3):.3f}",
        "3-Class Weighted": f"{cohen_kappa_score(a3, b3, weights='quadratic'):.3f}",
        "Binary Kappa": f"{cohen_kappa_score(ab, bb):.3f}",
        "Binary Agreement": f"{np.mean(ab == bb):.1%}",
        "3-Class Agreement": f"{np.mean(a3 == b3):.1%}",
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# --- Label distribution comparison ---
print(f"\n{'='*60}")
print("  LABEL DISTRIBUTION COMPARISON (Gold Set)")
print(f"{'='*60}\n")

dist_rows = []
for name, vals in [("Human (corrected)", corrected), ("Sonnet (gold-run)", sonnet_g), ("Sonnet (silver-run)", sonnet_s)]:
    total = len(vals)
    dist_rows.append({
        "Rater": name,
        "1-Noise": f"{(vals == 1).sum()} ({(vals == 1).mean():.1%})",
        "2-Directional": f"{(vals == 2).sum()} ({(vals == 2).mean():.1%})",
        "3-Actionable": f"{(vals == 3).sum()} ({(vals == 3).mean():.1%})",
    })

dist_df = pd.DataFrame(dist_rows)
print(dist_df.to_string(index=False))


Three-way overlap: 598 comments

                        Pair 3-Class Kappa 3-Class Weighted Binary Kappa Binary Agreement 3-Class Agreement
  Human vs Sonnet (gold-run)         0.469            0.602        0.869            97.0%             69.4%
Human vs Sonnet (silver-run)         0.457            0.592        0.869            97.0%             68.7%
Sonnet gold vs silver (self)         0.948            0.957        0.970            99.3%             98.0%

  LABEL DISTRIBUTION COMPARISON (Gold Set)

              Rater     1-Noise 2-Directional 3-Actionable
  Human (corrected) 231 (38.6%)   285 (47.7%)   82 (13.7%)
  Sonnet (gold-run)  66 (11.0%)   456 (76.3%)   76 (12.7%)
Sonnet (silver-run)  62 (10.4%)   460 (76.9%)   76 (12.7%)


In [ ]:
## ============================================================
## CELL 12: Silver Set Summary (Sonnet-only, n≈1800)
## ============================================================
## These comments were labeled only by Sonnet (silver run).
## No human labels exist for these.

import json
import pandas as pd

SONNET_SILVER_LABEL_PATH = "/content/drive/MyDrive/claude_labels_sonnet_silver.json"

with open(SONNET_SILVER_LABEL_PATH, "r") as f:
    sonnet_silver_labels = json.load(f)

# Split by set
gold_labels = {k: v for k, v in sonnet_silver_labels.items() if v.get("set") == "gold"}
silver_labels = {k: v for k, v in sonnet_silver_labels.items() if v.get("set") == "silver"}

print(f"Sonnet silver-run labels breakdown:")
print(f"  Gold set:   {len(gold_labels)}")
print(f"  Silver set: {len(silver_labels)}")
print(f"  Total:      {len(sonnet_silver_labels)}")

# Silver set label distribution
silver_vals = [v["label"] for v in silver_labels.values()]
print(f"\nSilver Set Label Distribution (Sonnet):")
print(f"  1 — Noise:       {silver_vals.count(1)} ({silver_vals.count(1)/len(silver_vals):.1%})")
print(f"  2 — Directional: {silver_vals.count(2)} ({silver_vals.count(2)/len(silver_vals):.1%})")
print(f"  3 — Actionable:  {silver_vals.count(3)} ({silver_vals.count(3)/len(silver_vals):.1%})")

# Compare distributions: gold vs silver
gold_sonnet_vals = [v["label"] for v in gold_labels.values()]
print(f"\nGold Set Label Distribution (Sonnet silver-run):")
print(f"  1 — Noise:       {gold_sonnet_vals.count(1)} ({gold_sonnet_vals.count(1)/len(gold_sonnet_vals):.1%})")
print(f"  2 — Directional: {gold_sonnet_vals.count(2)} ({gold_sonnet_vals.count(2)/len(gold_sonnet_vals):.1%})")
print(f"  3 — Actionable:  {gold_sonnet_vals.count(3)} ({gold_sonnet_vals.count(3)/len(gold_sonnet_vals):.1%})")

print(f"\nNote: Gold set was stratified (random + keyword + topic-enriched),")
print(f"so expect higher actionable rate there vs the purely random silver set.")


Sonnet silver-run labels breakdown:
  Gold set:   600
  Silver set: 1800
  Total:      2400

Silver Set Label Distribution (Sonnet):
  1 — Noise:       498 (27.7%)
  2 — Directional: 1235 (68.6%)
  3 — Actionable:  67 (3.7%)

Gold Set Label Distribution (Sonnet silver-run):
  1 — Noise:       62 (10.3%)
  2 — Directional: 462 (77.0%)
  3 — Actionable:  76 (12.7%)

Note: Gold set was stratified (random + keyword + topic-enriched),
so expect higher actionable rate there vs the purely random silver set.


In [ ]:
## ============================================================
## CELL 13: Save All Labeling Data (Gold + Silver) to Drive
## ============================================================
## Exports a master CSV with gold and silver labels, preserving
## all existing human + Sonnet gold-run data untouched.
## Silver set now uses Sonnet labels instead of Haiku.

import json
import os
import shutil
import pandas as pd

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/austin_survey_labels"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# --- Paths ---
LABEL_PATH               = "/content/drive/MyDrive/labeling_progress.json"
CLAUDE_V3_PATH           = "/content/drive/MyDrive/claude_labels_v3.json"
AUDIT_V3_PATH            = "/content/drive/MyDrive/audit_labels_v3.json"
SONNET_SILVER_LABEL_PATH = "/content/drive/MyDrive/claude_labels_sonnet_silver.json"

# --- Load everything ---
with open(LABEL_PATH, "r") as f:
    human_labels = json.load(f)

with open(CLAUDE_V3_PATH, "r") as f:
    sonnet_gold_labels = json.load(f)

with open(AUDIT_V3_PATH, "r") as f:
    audit_labels = json.load(f)

with open(SONNET_SILVER_LABEL_PATH, "r") as f:
    sonnet_silver_labels = json.load(f)

# --- Build corrected human labels (same as original) ---
corrected_human = {}
for k, v in human_labels.items():
    if k in audit_labels:
        corrected_human[k] = audit_labels[k]["audit_label"]
    else:
        corrected_human[k] = v["label"]

# --- Build master dataframe for ALL Sonnet-silver-labeled comments ---
rows = []
for k, sv in sonnet_silver_labels.items():
    idx = int(k)
    text = docs[idx] if idx < len(docs) else ""
    question = long_df.iloc[idx]["question_col"] if idx < len(long_df) else ""
    source_row = int(long_df.iloc[idx]["source_row"]) if idx < len(long_df) else -1

    is_gold = sv.get("set") == "gold"

    row = {
        "idx": idx,
        "source_row": source_row,
        "question_col": question,
        "comment_text": text,
        "set": sv.get("set", "unknown"),
        # Human labels (gold only)
        "human_label": human_labels.get(k, {}).get("label") if is_gold else None,
        "human_corrected": corrected_human.get(k) if is_gold else None,
        # Sonnet gold-run labels (gold only)
        "sonnet_gold_label": sonnet_gold_labels.get(k, {}).get("label") if is_gold else None,
        "sonnet_gold_justification": sonnet_gold_labels.get(k, {}).get("justification", "") if is_gold else "",
        # Sonnet silver-run labels (all 2400)
        "sonnet_label": sv.get("label"),
        "sonnet_justification": sv.get("justification", ""),
        # Metadata
        "was_audited": k in audit_labels if is_gold else False,
        "sample_source": sv.get("source", ""),
        "topic": sv.get("topic"),
        # Binary convenience columns
        "binary_human_corrected": (1 if corrected_human.get(k) == 3 else 0) if is_gold else None,
        "binary_sonnet_gold": (1 if sonnet_gold_labels.get(k, {}).get("label") == 3 else 0) if is_gold else None,
        "binary_sonnet": 1 if sv.get("label") == 3 else 0,
    }
    rows.append(row)

master_df = pd.DataFrame(rows).sort_values("idx").reset_index(drop=True)

# --- Save ---
master_path = os.path.join(DRIVE_OUTPUT_DIR, "labeled_comments_gold_silver.csv")
master_df.to_csv(master_path, index=False)

# Save full unlabeled dataset
full_df = long_df.copy()
full_df["comment_clean"] = docs
full_path = os.path.join(DRIVE_OUTPUT_DIR, "all_comments_clean.csv")
full_df.to_csv(full_path, index=False)

# Copy raw JSON files
for name, path in [
    ("human_labels.json", LABEL_PATH),
    ("claude_labels_v3_sonnet_gold.json", CLAUDE_V3_PATH),
    ("audit_labels_v3.json", AUDIT_V3_PATH),
    ("claude_labels_sonnet_silver.json", SONNET_SILVER_LABEL_PATH),
]:
    if os.path.exists(path):
        shutil.copy(path, os.path.join(DRIVE_OUTPUT_DIR, name))

# --- Summary ---
gold_df = master_df[master_df["set"] == "gold"]
silver_df = master_df[master_df["set"] == "silver"]

print(f"Saved to: {DRIVE_OUTPUT_DIR}\n")
print(f"  labeled_comments_gold_silver.csv — {len(master_df)} total rows")
print(f"    Gold set:   {len(gold_df)} (human + Sonnet gold-run + Sonnet silver-run labels)")
print(f"    Silver set: {len(silver_df)} (Sonnet labels only)")
print(f"\n  Gold set columns: human_label, human_corrected, sonnet_gold_label, sonnet_label")
print(f"  Silver set columns: sonnet_label (human/sonnet_gold columns are null)")
print(f"\n  Raw JSON files copied for provenance")
print(f"\n  Actionable rate by set:")
print(f"    Gold (human corrected):    {gold_df['binary_human_corrected'].mean():.1%}")
print(f"    Gold (Sonnet gold-run):    {gold_df['binary_sonnet_gold'].mean():.1%}")
print(f"    Gold (Sonnet silver-run):  {gold_df['binary_sonnet'].mean():.1%}")
print(f"    Silver (Sonnet):           {silver_df['binary_sonnet'].mean():.1%}")


Saved to: /content/drive/MyDrive/austin_survey_labels

  labeled_comments_gold_silver.csv — 2400 total rows
    Gold set:   600 (human + Sonnet gold-run + Sonnet silver-run labels)
    Silver set: 1800 (Sonnet labels only)

  Gold set columns: human_label, human_corrected, sonnet_gold_label, sonnet_label
  Silver set columns: sonnet_label (human/sonnet_gold columns are null)

  Raw JSON files copied for provenance

  Actionable rate by set:
    Gold (human corrected):    13.8%
    Gold (Sonnet gold-run):    12.7%
    Gold (Sonnet silver-run):  12.7%
    Silver (Sonnet):           3.7%


In [ ]:
## ============================================================
## CELL 14: Where did the 3s come from? (Gold + Silver)
## ============================================================

print("GOLD SET — Actionable rate by sample source (Sonnet silver-run):")
gold_df = master_df[master_df["set"] == "gold"]
counts = gold_df[gold_df["binary_sonnet"] == 1].groupby("sample_source").size()
totals = gold_df.groupby("sample_source").size()
rates = (counts / totals * 100).round(1)
summary = pd.DataFrame({"actionable": counts, "total": totals, "rate_%": rates})
print(summary.sort_values("rate_%", ascending=False))

print(f"\nSILVER SET — Actionable rate (Sonnet):")
silver_df = master_df[master_df["set"] == "silver"]
print(f"  Actionable: {silver_df['binary_sonnet'].sum()} / {len(silver_df)} ({silver_df['binary_sonnet'].mean():.1%})")


GOLD SET — Actionable rate by sample source (Sonnet silver-run):
               actionable  total  rate_%
sample_source                           
topic                  34    200    17.0
keyword                33    200    16.5
random                  9    200     4.5

SILVER SET — Actionable rate (Sonnet):
  Actionable: 67 / 1800 (3.7%)


# AI Citation Section

Cleaned up formatting of print statements, used to generate facy print tables to better present results to future readers, Sonnet 4.6, Anthropic

Used to improve labeling prompt legibility, Sonnet 4.6, Anthropic

Typeahead used in some cases, Google Colab - Gemini, Google